# SaaS Pricing Optimization: A/B Test Analysis
**Goal:** Evaluate the impact of a new pricing strategy on revenue and conversion using statistical testing and bootstrapping.

---
## Summary of Results:
- **ARPU Mean Difference:** 7.76
- **95% Confidence Interval:** [1.52, 14.00]
- **Decision:** **Statistical significance confirmed. Recommend rollout.**
---

In [2]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
np.random.seed(42)

n_users = 5000
groups = ['A', 'B']

data = pd.DataFrame({
    'user_id': range(n_users),
    'group': np.random.choice(groups, n_users)
})

def simulate_revenue(group):
    if group == 'A':
        converted = np.random.binomial(1, 0.05)
        revenue = converted * np.random.normal(500, 100) if converted else 0
    else:
        converted = np.random.binomial(1, 0.058)
        revenue = converted * np.random.normal(510, 110) if converted else 0
    return pd.Series([converted, max(0, revenue)])

data[['converted', 'revenue']] = data['group'].apply(simulate_revenue)

print(data.groupby('group').agg({'user_id': 'count', 'converted': 'mean', 'revenue': 'mean'}))

       user_id  converted    revenue
group                               
A         2504   0.044728  22.183358
B         2496   0.060897  29.898526


In [4]:
from statsmodels.stats.proportion import proportions_ztest

conversions = data.groupby('group')['converted'].sum()
nobs = data.groupby('group')['converted'].count()

z_stat, p_val_conv = proportions_ztest(conversions[::-1], nobs[::-1])

print(f"Z-statistic for Conversion: {z_stat:.4f}")
print(f"P-value for Conversion: {p_val_conv:.4f}")

Z-statistic for Conversion: 2.5562
P-value for Conversion: 0.0106


In [5]:
rev_a = data[data['group'] == 'A']['revenue']
rev_b = data[data['group'] == 'B']['revenue']

u_stat, p_val_rev = stats.mannwhitneyu(rev_a, rev_b, alternative='two-sided')

print(f"U-statistic for Revenue: {u_stat:.4f}")
print(f"P-value for Revenue: {p_val_rev:.4f}")

U-statistic for Revenue: 3074560.0000
P-value for Revenue: 0.0108


In [11]:
def bootstrap_arpu(group_a, group_b, iterations=10000):
    boot_means_diff = []
    for _ in range(iterations):
        boot_a = group_a.sample(frac=1, replace=True).mean()
        boot_b = group_b.sample(frac=1, replace=True).mean()
        boot_means_diff.append(boot_b - boot_a)
    return pd.Series(boot_means_diff)

diffs = bootstrap_arpu(rev_a, rev_b)
ci_low = diffs.quantile(0.025)
ci_high = diffs.quantile(0.975)
mean_diff = diffs.mean()

print(f"--- Bootstrap Results (ARPU) ---")
print(f"Mean Difference: {mean_diff:.2f}")
print(f"95% Confidence Interval: [{ci_low:.2f}, {ci_high:.2f}]")

if ci_low > 0:
    print("\nConclusion: Statistically significant positive growth. Recommend rollout.")
elif ci_high < 0:
    print("\nConclusion: Statistically significant negative impact. Do NOT rollout.")
else:
    print("\nConclusion: No statistically significant difference. Results could be due to chance.")

--- Bootstrap Results (ARPU) ---
Mean Difference: 7.76
95% Confidence Interval: [1.52, 14.00]

Conclusion: Statistically significant positive growth. Recommend rollout.


In [9]:
shapiro_a = stats.shapiro(rev_a[rev_a > 0])
shapiro_b = stats.shapiro(rev_b[rev_b > 0])

print(f"Shapiro-Wilk test P-value (Group A): {shapiro_a.pvalue:.4f}")
print(f"Shapiro-Wilk test P-value (Group B): {shapiro_b.pvalue:.4f}")

t_stat, p_val_t = stats.ttest_ind(rev_a, rev_b, equal_var=False) # Welch's t-test

print(f"\nT-statistic: {t_stat:.4f}")
print(f"T-test P-value: {p_val_t:.4f}")

Shapiro-Wilk test P-value (Group A): 0.6024
Shapiro-Wilk test P-value (Group B): 0.4371

T-statistic: -2.4181
T-test P-value: 0.0156
